In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="taresh18/AnimeVox", 
                  repo_type="dataset", local_dir="./AnimeVox")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:03<00:00,  2.33it/s]


'/home/ubuntu/AnimeVox'

In [3]:
files = glob('AnimeVox/*/*.parquet')
files

['AnimeVox/data/train-00006-of-00007.parquet',
 'AnimeVox/data/train-00000-of-00007.parquet',
 'AnimeVox/data/train-00001-of-00007.parquet',
 'AnimeVox/data/train-00004-of-00007.parquet',
 'AnimeVox/data/train-00003-of-00007.parquet',
 'AnimeVox/data/train-00005-of-00007.parquet',
 'AnimeVox/data/train-00002-of-00007.parquet']

In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['character_name'].iloc[i]}"
            })
        
    return data

In [6]:
data = loop((files[:1], 0))
data

  0%|          | 0/1574 [00:00<?, ?it/s]


[{'audio_filename': 'AnimeVox_audio/AnimeVox-data-train-00006-of-00007_0.mp3',
  'text': 'How much do you know about the Subaru that I see? The one I was just telling you about?',
  'speaker': 'AnimeVox_audio_Rem'}]

In [8]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1575/1575 [00:57<00:00, 27.61it/s]


In [9]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'AnimeVox_audio/AnimeVox-data-train-00006-of-00007_0.mp3',
 'text': 'How much do you know about the Subaru that I see? The one I was just telling you about?',
 'speaker': 'AnimeVox_audio_Rem'}

In [10]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'AnimeVox')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 227.86ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  442kB /  442kB, 51.4kB/s  
Processing Files (1 / 1): 100%|██████████|  442kB /  442kB, 50.3kB/s  
New Data Upload: 100%|██████████|  442kB /  442kB, 50.3kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.12s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/20fcfb99e91900c6bd58cbea4c31baf59b5191dd', commit_message='Upload dataset', commit_description='', oid='20fcfb99e91900c6bd58cbea4c31baf59b5191dd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('AnimeVox-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [13]:
!zip -rq AnimeVox_audio.zip AnimeVox_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS AnimeVox_audio.zip --repo-type=dataset

In [17]:
!zip -rq AnimeVox_audio_neucodec.zip AnimeVox_audio_neucodec

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS AnimeVox_audio_neucodec.zip --repo-type=dataset